# Analyze Embedding Model for Semantically Similar Values

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd


/scratch/gf2467/conda/envs/semsketch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mpnet_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1627.54it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
gemma_model = SentenceTransformer("google/embeddinggemma-300m")

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 1662.22it/s, Materializing param=norm.weight]                                


In [4]:
semantically_similar_values = [
    ["AT&T Stadium", "AT&T Park"],
    ["AT&T Stadium", "Cowboys Stadium"],
    ["AT&T Stadium", "jones AT&T stadium"],
]

In [6]:
from IPython.display import display, HTML

import matplotlib
import matplotlib.pyplot as plt

for val_list in semantically_similar_values:
    # Get embeddings from sentence transformer
    mpnet_embeddings = mpnet_model.encode(val_list, convert_to_numpy=True, show_progress_bar=False)
    gemma_embeddings = gemma_model.encode(val_list, convert_to_numpy=True, show_progress_bar=False)

    for model_name, embeddings in [("mpnet", mpnet_embeddings), ("gemma", gemma_embeddings)]:
        # Ensure embeddings are float32 and normalized (for cosine similarity)
        embeddings = embeddings.astype(np.float32)
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1, norms)
        emb_norm = embeddings / norms

        # Compute all unique value pairs and their cosine similarities
        pair_sims = []
        for i in range(len(embeddings)):
            for j in range(i+1, len(embeddings)):
                cos_sim = float(emb_norm[i].dot(emb_norm[j]))
                pair_sims.append((val_list[i], val_list[j], cos_sim))
        # Sort pairs by cosine similarity in descending order
        pair_sims.sort(key=lambda x: x[2], reverse=True)

        # Display as a pretty table in Jupyter notebook
        df = pd.DataFrame(pair_sims, columns=["Value 1", "Value 2", "Cosine Similarity"])

        # Debug fix: ensure matplotlib is imported for background_gradient to work
        try:
            display(HTML(f"<h4>Cosine similarity table for {model_name}: {val_list}</h4>"))
            display(df.style.format({f"Model: {model_name} Cosine Similarity": "{:.3f}"}).background_gradient(subset=["Cosine Similarity"], cmap="Blues"))
        except ImportError as e:
            print(f"⚠️ Could not apply background gradient to DataFrame (requires matplotlib): {e}")
            display(HTML(f"<h4>Cosine similarity table for {model_name}: {val_list}</h4>"))
            display(df)

,Value 1,Value 2,Cosine Similarity
0,AT&T Stadium,AT&T Park,0.902439


,Value 1,Value 2,Cosine Similarity
0,AT&T Stadium,AT&T Park,0.895839


,Value 1,Value 2,Cosine Similarity
0,AT&T Stadium,Cowboys Stadium,0.773428


,Value 1,Value 2,Cosine Similarity
0,AT&T Stadium,Cowboys Stadium,0.786464


,Value 1,Value 2,Cosine Similarity
0,AT&T Stadium,jones AT&T stadium,0.745121


,Value 1,Value 2,Cosine Similarity
0,AT&T Stadium,jones AT&T stadium,0.868951


In [ ]:
# import test dataset
import pandas as pd
from tqdm import tqdm

# Load the test dataset (expects columns: title_l, title_r) from AutoFJ (SportsFacility)
df_test = pd.read_csv("test_entity_embedding_model.csv")

results = []

for idx, row in tqdm(df_test.iterrows(), total=len(df_test)):
    val_list = [str(row["title_l"]), str(row["title_r"])]

    # Get embeddings from both models
    mpnet_embeddings = mpnet_model.encode(val_list, convert_to_numpy=True, show_progress_bar=False)
    
    # Obtain Gemma embeddings once at full size, then truncate to dimensions
    full_emb = gemma_model.encode(val_list, convert_to_numpy=True, show_progress_bar=False)
    gemma_embeddings_dict = {}
    for dim in [128, 256, 512, 768]:
        gemma_emb_dim = full_emb[:, :dim]
        gemma_embeddings_dict[dim] = gemma_emb_dim

    # Normalize embeddings for cosine similarity
    def normalize(emb):
        norms = np.linalg.norm(emb, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1, norms)
        return emb / norms

    mpnet_norm = normalize(mpnet_embeddings)
    # Compute cosine similarity (since there are 2 values, one score)
    mpnet_cos_sim = float(mpnet_norm[0].dot(mpnet_norm[1]))

    # Compute cosine similarities for each Gemma dimension
    gemma_cos_sims = {}
    for dim, emb in gemma_embeddings_dict.items():
        normed = normalize(emb)
        cos_sim = float(normed[0].dot(normed[1]))
        gemma_cos_sims[dim] = cos_sim

    row_result = {
        "title_l": val_list[0],
        "title_r": val_list[1],
        "mpnet_cosine_similarity": mpnet_cos_sim,
    }
    for dim in [128, 256, 512, 768]:
        row_result[f"gemma_cosine_similarity_{dim}"] = gemma_cos_sims[dim]
    # If you still want the old column:
    row_result["gemma_cosine_similarity"] = gemma_cos_sims[768]
    if abs(row_result["mpnet_cosine_similarity"] - row_result["gemma_cosine_similarity"]) > 0.1:
        results.append(row_result)


# Display as dataframe, color gradient on cosine similarity
df_results = pd.DataFrame(results)
from IPython.display import display, HTML

gemma_cols = [f"gemma_cosine_similarity_{dim}" for dim in [128, 256, 512, 768]]
cols_to_format = ["mpnet_cosine_similarity"] + gemma_cols + ["gemma_cosine_similarity"]

display(HTML("<h4>Cosine similarity on test dataset ('test_entity_embedding_model.csv')</h4>"))
display(
    df_results.style
        .format({col: "{:.3f}" for col in cols_to_format})
        .background_gradient(subset=cols_to_format, cmap="Blues")
)


100%|██████████| 672/672 [42:02<00:00,  3.75s/it]


,title_l,title_r,mpnet_cosine_similarity,gemma_cosine_similarity_128,gemma_cosine_similarity_256,gemma_cosine_similarity_512,gemma_cosine_similarity_768,gemma_cosine_similarity
0,Bell Centre,Centre Bell,0.698,0.928,0.913,0.894,0.881,0.881
1,Rose Garden (arena),Moda Center,0.310,0.595,0.536,0.490,0.428,0.428
2,Reliant Stadium,NRG Stadium,0.634,0.831,0.817,0.780,0.761,0.761
3,Khasbag,Khasbag Wrestling Stadium,0.575,0.841,0.795,0.756,0.732,0.732
4,Marista Hall,Chevrolet Hall (Belo Horizonte),0.364,0.613,0.580,0.526,0.507,0.507
5,"Fartown Ground, Huddersfield",Fartown Ground,0.627,0.896,0.887,0.865,0.854,0.854
6,Columbus Crew Stadium,Mapfre Stadium,0.615,0.834,0.792,0.750,0.733,0.733
7,LP Field,Nissan Stadium,0.177,0.674,0.611,0.527,0.480,0.480
8,The Home Depot Center,StubHub Center,0.412,0.838,0.784,0.770,0.754,0.754
9,Tampa Bay Times Forum,Amalie Arena,0.401,0.730,0.655,0.555,0.519,0.519
